In [5]:
import pandas as pd, numpy as np, faiss, joblib
from sentence_transformers import SentenceTransformer

kb = pd.read_csv("kb.csv")
print("Knowledge base entries:", len(kb))

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = (kb["title"] + ". " + kb["answer"]).tolist()
vecs = embedder.encode(texts, normalize_embeddings=True).astype("float32")

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)
faiss.write_index(index, "kb.index")
print("Index built with", index.ntotal, "entries")

Knowledge base entries: 38


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2137.93it/s]


Index built with 38 entries


In [6]:
def search(query, k=3):
    v = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(v, k)
    return [(kb.iloc[i], float(s)) for s, i in zip(scores[0], ids[0])]

In [7]:
questions = [
    "i want my money back",
    "how long does a refund take",
    "my card was charged twice",
    "where is my parcel",
    "i forgot my password",
]
for q in questions:
    print("\nQ:", q)
    for row, score in search(q, 3):
        print(f"   {score:.2f}  {row['title']}")
        


Q: i want my money back
   0.53  Money back for a cancelled order
   0.52  Refund status
   0.48  Where is my refund

Q: how long does a refund take
   0.76  Refund timeline
   0.67  Refund eligibility
   0.60  Money back for a cancelled order

Q: my card was charged twice
   0.62  Charged twice
   0.31  Account locked
   0.31  Refund status

Q: where is my parcel
   0.60  Track my parcel
   0.49  Order not received yet
   0.47  Where is my order

Q: i forgot my password
   0.63  Password reset
   0.58  Account locked
   0.50  Login problems


In [8]:
outside = [
    "do you sell shoes",
    "can i get a student discount",
    "do you have a physical store",
    "do you offer gift wrapping",
    "what is the weather today",
]
for q in outside:
    top = search(q, 1)[0]
    print(f"{top[1]:.2f}  {q}   (closest: {top[0]['title']})")

0.25  do you sell shoes   (closest: Item out of stock)
0.28  can i get a student discount   (closest: Non-refundable items)
0.27  do you have a physical store   (closest: International shipping)
0.42  do you offer gift wrapping   (closest: Non-refundable items)
0.12  what is the weather today   (closest: Order not received yet)


In [9]:
import ollama

clf = joblib.load("intent_model.pkl")

MODEL_NAME = "llama3.2:3b"      # change to llama3.2:1b if you downloaded that one
MIN_SCORE = 0.45                # below this, no policy is relevant enough
SENSITIVE = ["lawyer", "legal", "fraud", "scam", "chargeback", "sue",
             "police", "complaint", "court", "cheated", "harass"]

def handle(query):
    q = query.lower().strip()

    # 1. classify
    proba = clf.predict_proba([q])[0]
    intent = clf.classes_[proba.argmax()]
    conf = float(proba.max())

    # 2. search
    hits = search(query, 3)
    best = hits[0][1]

    # 3. decide whether a human is needed
    reasons = []
    if best < MIN_SCORE:
        reasons.append("no relevant policy found")
    if conf < 0.35 and best < 0.55:
        reasons.append("low confidence in both intent and evidence")
    if any(w in q for w in SENSITIVE):
        reasons.append("sensitive topic")

    result = {
        "query": query,
        "intent": intent,
        "confidence": round(conf, 2),
        "evidence": [r["title"] for r, _ in hits],
        "best_score": round(best, 2),
    }

    if reasons:
        result.update(status="ESCALATE", reply="", reasons=reasons)
        return result

    # 4. ask the AI model to answer ONLY from the evidence
    context = "\n".join(f"- {r['title']}: {r['answer']}" for r, _ in hits)
    prompt = f"""You are a polite customer support agent.
Answer the customer's question using ONLY the policy facts below.
Do not add any information that is not in the facts.
If the facts do not answer the question, reply with exactly: NOT_ENOUGH_INFO

Policy facts:
{context}

Customer question: {query}

Write a reply of 2 to 3 short sentences."""

    out = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )["message"]["content"].strip()

    if "NOT_ENOUGH_INFO" in out:
        result.update(status="ESCALATE", reply="",
                      reasons=["model could not answer from the evidence"])
    else:
        result.update(status="DRAFT_READY", reply=out, reasons=[])
    return result

In [10]:
tests = [
    "How long does a refund take?",
    "where is my parcel",
    "my card was charged twice",
    "I was cheated and I want to sue you",
    "do you sell shoes",
    "do you offer gift wrapping",
    "can i pay with bitcoin",
]
for t in tests:
    r = handle(t)
    print("\nQ:", r["query"])
    print("  intent:", r["intent"], "| confidence:", r["confidence"],
          "| search score:", r["best_score"])
    print("  status:", r["status"], r["reasons"])
    print("  reply:", r["reply"])


Q: How long does a refund take?
  intent: refund | confidence: 0.87 | search score: 0.76
  status: DRAFT_READY []
  reply: Approved refunds are credited to the original payment method within 5 to 7 business days.

Q: where is my parcel
  intent: delivery | confidence: 0.41 | search score: 0.6
  status: DRAFT_READY []
  reply: You can see the location of your order under My Orders and then Track Order. A tracking link is also sent by email and SMS once it is dispatched.

Q: my card was charged twice
  intent: account | confidence: 0.28 | search score: 0.62
  status: DRAFT_READY []
  reply: To help with your issue, please send us both transaction IDs so we can look into the extra charge. We will refund the extra amount within 5 to 7 business days.

Q: I was cheated and I want to sue you
  intent: account | confidence: 0.29 | search score: 0.29
  status: ESCALATE ['no relevant policy found', 'low confidence in both intent and evidence', 'sensitive topic']
  reply: 

Q: do you sell shoes


In [11]:
for t in ["do you offer gift wrapping", "can i get a student discount", "do you have a physical store"]:
    r = handle(t)
    print("\nQ:", t)
    print("  search score:", r["best_score"], "| status:", r["status"], r["reasons"])
    print("  reply:", r["reply"])


Q: do you offer gift wrapping
  search score: 0.42 | status: ESCALATE ['no relevant policy found']
  reply: 

Q: can i get a student discount
  search score: 0.28 | status: ESCALATE ['no relevant policy found']
  reply: 

Q: do you have a physical store
  search score: 0.27 | status: ESCALATE ['no relevant policy found', 'low confidence in both intent and evidence']
  reply: 
